# PDF to Text
### Gemini prompt 22 Feb 2025: "python code to take pdf invoices as input and return name of saupplier, value and date using machine learning"

In [3]:
import io
import re
from datetime import datetime

import fitz  # PyMuPDF
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Download necessary NLTK resources (run once)
nltk.download('stopwords', quiet=True)

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    text = ""
    try:
        with fitz.open(pdf_path) as pdf_document:
            for page_num in range(pdf_document.page_count):
                page = pdf_document[page_num]
                text += page.get_text()
    except Exception as e:
        print(f"Error extracting text: {e}")
    return text

def preprocess_text(text):
    """Preprocesses text for NLP."""
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # Remove special characters
    stop_words = set(stopwords.words("english"))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

def train_supplier_classifier(training_data):
    """Trains a classifier to identify supplier names."""
    texts = [preprocess_text(item["text"]) for item in training_data]
    labels = [item["supplier"] for item in training_data]

    vectorizer = TfidfVectorizer()
    features = vectorizer.fit_transform(texts)

    classifier = LogisticRegression()
    classifier.fit(features, labels)

    return vectorizer, classifier

def extract_supplier(text, vectorizer, classifier):
    """Extracts supplier name using the trained classifier."""
    processed_text = preprocess_text(text)
    features = vectorizer.transform([processed_text])
    try:
        supplier = classifier.predict(features)[0]
        return supplier
    except:
        return None

def extract_value(text):
    """Extracts invoice value using regular expressions."""
    value_match = re.search(r"(?:total|amount|grand total|invoice total)\s*[:\$£€]?\s*([\d,.]+)", text, re.IGNORECASE)
    if value_match:
        value_str = value_match.group(1).replace(",", "")
        try:
            value = float(value_str)
            return value
        except ValueError:
            return None
    return None

def extract_date(text):
    """Extracts invoice date using regular expressions."""
    date_patterns = [
        r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}",  # DD/MM/YYYY, MM/DD/YYYY, etc.
        r"[a-zA-Z]{3,9}\s+\d{1,2},\s+\d{4}",  # Month DD, YYYY
        r"\d{4}[/-]\d{1,2}[/-]\d{1,2}", #YYYY-MM-DD
        r"\d{1,2}\s+[a-zA-Z]{3,9}\s+\d{4}" # DD Month YYYY
    ]

    for pattern in date_patterns:
        date_match = re.search(pattern, text)
        if date_match:
            date_str = date_match.group(0)
            try:
                date = datetime.strptime(date_str, "%d/%m/%Y") #add other formats as needed.
                return date
            except ValueError:
                try:
                    date = datetime.strptime(date_str, "%m/%d/%Y")
                    return date
                except ValueError:
                    try:
                        date = datetime.strptime(date_str, "%B %d, %Y")
                        return date
                    except ValueError:
                        try:
                            date = datetime.strptime(date_str, "%Y-%m-%d")
                            return date
                        except ValueError:
                            try:
                                date = datetime.strptime(date_str, "%d %B %Y")
                                return date
                            except ValueError:
                                pass #try next pattern
    return None

def extract_invoice_info(pdf_path, vectorizer, classifier):
    """Extracts supplier, value, and date from an invoice."""
    text = extract_text_from_pdf(pdf_path)
    if not text:
        return None

    supplier = extract_supplier(text, vectorizer, classifier)
    value = extract_value(text)
    date = extract_date(text)

    return {"supplier": supplier, "value": value, "date": date}

# Example Usage (with training data):
training_data = [
    {"text": "Invoice from Acme Corp. Total: $100", "supplier": "Acme Corp."},
    {"text": "Bill from Global Services Ltd. Amount: £200", "supplier": "Global Services Ltd."},
    {"text": "Invoice from Widgets Inc. Grand Total: €150", "supplier": "Widgets Inc."},
    {"text": "Payment from Example Company", "supplier": "Example Company"},
    # Add more training data...
]

vectorizer, classifier = train_supplier_classifier(training_data)

invoice_path = r'C:\Users\hilton.netta\Downloads\Scan from 2025-02-10 08_33_50 AM.pdf'  # Replace with your invoice path
invoice_info = extract_invoice_info(invoice_path, vectorizer, classifier)

if invoice_info:
    print(f"Supplier: {invoice_info['supplier']}")
    print(f"Value: {invoice_info['value']}")
    print(f"Date: {invoice_info['date']}")
else:
    print("Could not extract invoice information.")

Could not extract invoice information.
